In [46]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
SS_VER = 'SS_VER_2_1'

In [47]:
%run "../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍


In [41]:
import subprocess
from IPython.display import HTML, display
import html

cmd = f"""
cd ../../firmware/mcu/simpleserial-xor
make PLATFORM={PLATFORM} SS_VER={SS_VER} CRYPTO_TARGET=MBEDTLS CRYPTO_OPTIONS=AES128C -j
"""

result = subprocess.run( cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

display(HTML(f"""
<div style="max-height: 300px; overflow-y: auto; border: 1px solid #888; padding: 8px; background-color: #111; color: #ddd; font-family: monospace;
    white-space: pre-wrap;">
{html.escape(result.stdout)}
</div>
"""))

In [42]:
cw.program_target(scope, prog, "../../firmware/mcu/simpleserial-xor/simpleserial-xor-{}.hex".format(PLATFORM))

In [48]:
def hexstr(data):
    if data is None:
        return "None"
    return " ".join(f"{b:02X}" for b in data)


In [51]:
# simple xor

ktp = cw.ktp.Basic()
key, text = ktp.next()

target.flush()

target.simpleserial_write('b', key)
ack = target.simpleserial_wait_ack()

target.simpleserial_write('a', text)

response = target.simpleserial_read('r', 16)

expected = bytearray(t ^ k for t, k in zip(text, key))

print("received:", hexstr(response))
print("expected:", hexstr(expected))
print("match   :", response == expected)

received: 0E CC BF E0 0B 9B 92 CB 1D DC BB 90 E5 6F 9E AE
expected: 0E CC BF E0 0B 9B 92 CB 1D DC BB 90 E5 6F 9E AE
match   : True


In [57]:
import chipwhisperer as cw
from Crypto.Cipher import AES

# Generate key and plaintext
ktp = cw.ktp.Basic()
key, _ = ktp.next()

# Initial CTR counter
counter = bytearray([0x00] * 16)

plaintext = bytearray(range(1, 33))

target.flush()

# Set AES key
target.simpleserial_write('b', key)
ack = target.simpleserial_wait_ack()

# Set counter 
target.simpleserial_write('e', counter)
ack = target.simpleserial_wait_ack()

# Encrypt plaintext on SAM4S
target.simpleserial_write('c', plaintext)

response = target.simpleserial_read('r', 32)


# --------------------------------------------------
# Calculate expected AES-128-CTR result on the PC
# --------------------------------------------------

# CTR encryption:
#
# ciphertext = plaintext XOR AES(key, counter)
#
aes = AES.new(bytes(key), AES.MODE_ECB)

keystream = aes.encrypt(counter)

expected = bytearray(
    p ^ k
    for p, k in zip(plaintext, keystream)
)


print("key      :", hexstr(key))
print("counter  :", hexstr(counter))
print("plaintext:", hexstr(text))
print("keystream:", hexstr(keystream))
print("received :", hexstr(response))
print("expected :", hexstr(expected))
print("match    :", response == expected)

key      : 2B 7E 15 16 28 AE D2 A6 AB F7 15 88 09 CF 4F 3C
counter  : 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00
plaintext: 25 B2 AA F6 23 35 40 6D B6 2B AE 18 EC A0 D1 92
keystream: 7D F7 6B 0C 1A B8 99 B3 3E 42 F0 47 B9 1B 54 6F
received : 7C F5 68 08 1F BE 9E BB 37 48 FB 4B B4 15 5B 7F 46 00 6E 54 21 A7 A9 A7 B7 EE 7D A5 DA 6C 70 E6
expected : 7C F5 68 08 1F BE 9E BB 37 48 FB 4B B4 15 5B 7F
match    : False
